In [1]:
from qiskit import QuantumCircuit

In [2]:
def baseline_adder_subtractor_core():
    """
    Baseline reversible quantum adder-cum-subtractor.

    q0 = A
    q1 = B
    q2 = Mode
         0 -> Addition
         1 -> Subtraction
    q3 = Result
    q4 = Carry/Borrow
    """

    qc = QuantumCircuit(5)

    # ------------------------------------------------
    # Step 1: B' = B XOR Mode
    # ------------------------------------------------
    qc.cx(2, 1)

    # ------------------------------------------------
    # Step 2: Result = A XOR B' XOR Mode
    # ------------------------------------------------
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # ------------------------------------------------
    # Step 3: Carry calculation of full adder
    # Carry = A.B' XOR A.Mode XOR B'.Mode
    # ------------------------------------------------
    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)

    # ------------------------------------------------
    # Step 4:
    # Mode=0 -> Carry
    # Mode=1 -> Borrow = NOT Carry
    # ------------------------------------------------
    qc.cx(2, 4)

    # ------------------------------------------------
    # Step 5: Restore original B
    # ------------------------------------------------
    qc.cx(2, 1)

    return qc

In [3]:
qc = baseline_adder_subtractor_core()

print(qc)

                                                  
q_0: ───────■──────────────■────■─────────────────
     ┌───┐  │              │    │            ┌───┐
q_1: ┤ X ├──┼────■─────────■────┼────■───────┤ X ├
     └─┬─┘  │    │         │    │    │       └─┬─┘
q_2: ──■────┼────┼────■────┼────■────■────■────■──
          ┌─┴─┐┌─┴─┐┌─┴─┐  │    │    │    │       
q_3: ─────┤ X ├┤ X ├┤ X ├──┼────┼────┼────┼───────
          └───┘└───┘└───┘┌─┴─┐┌─┴─┐┌─┴─┐┌─┴─┐     
q_4: ────────────────────┤ X ├┤ X ├┤ X ├┤ X ├─────
                         └───┘└───┘└───┘└───┘     


In [4]:
fig = qc.draw("mpl", fold=-1)

fig.savefig(
    "../results/circuits/baseline_adder_subtractor.png",
    dpi=300,
    bbox_inches="tight"
)

In [5]:
from qiskit import QuantumCircuit

def test_adder_subtractor(a, b, mode):

    qc = QuantumCircuit(5, 5)

    # Prepare A
    if a == 1:
        qc.x(0)

    # Prepare B
    if b == 1:
        qc.x(1)

    # Prepare Mode
    if mode == 1:
        qc.x(2)

    # ---------------------------------------------
    # B' = B XOR Mode
    # ---------------------------------------------
    qc.cx(2, 1)

    # ---------------------------------------------
    # Result = A XOR B' XOR Mode
    # ---------------------------------------------
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # ---------------------------------------------
    # Carry calculation
    # ---------------------------------------------
    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)

    # ---------------------------------------------
    # Carry -> Borrow conversion when Mode = 1
    # ---------------------------------------------
    qc.cx(2, 4)

    # ---------------------------------------------
    # Restore B
    # ---------------------------------------------
    qc.cx(2, 1)

    # Measurement
    qc.measure(
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]
    )

    return qc

In [6]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

qc = test_adder_subtractor(1, 1, 0)

result = simulator.run(qc, shots=1000).result()

counts = result.get_counts()

print(counts)

{'10011': 1000}


In [7]:
qc = test_adder_subtractor(1, 1, 1)

result = simulator.run(qc, shots=1000).result()

counts = result.get_counts()

print(counts)

{'00111': 1000}


In [8]:
test_cases = [
    (0, 0, 0),
    (0, 1, 0),
    (1, 0, 0),
    (1, 1, 0),

    (0, 0, 1),
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 1)
]

print("A B Mode | Operation | Result | Carry/Borrow")
print("------------------------------------------------")

for a, b, mode in test_cases:

    qc = test_adder_subtractor(a, b, mode)

    result = simulator.run(qc, shots=100).result()

    counts = result.get_counts()

    measured = list(counts.keys())[0]

    # Qiskit displays q4 q3 q2 q1 q0
    carry_borrow = int(measured[0])
    result_bit = int(measured[1])

    if mode == 0:
        operation = "ADD"
    else:
        operation = "SUB"

    print(
        f"{a} {b}   {mode}   | "
        f"{operation:8} | "
        f"   {result_bit}   | "
        f"     {carry_borrow}"
    )

A B Mode | Operation | Result | Carry/Borrow
------------------------------------------------
0 0   0   | ADD      |    0   |      0
0 1   0   | ADD      |    1   |      0
1 0   0   | ADD      |    1   |      0
1 1   0   | ADD      |    0   |      1
0 0   1   | SUB      |    0   |      0
0 1   1   | SUB      |    1   |      1
1 0   1   | SUB      |    1   |      0
1 1   1   | SUB      |    0   |      0


In [9]:
print("Number of qubits:", qc.num_qubits)
print("Circuit depth:", qc.depth())
print("Gate count:", qc.count_ops())

Number of qubits: 5
Circuit depth: 9
Gate count: OrderedDict({'cx': 6, 'measure': 5, 'x': 3, 'ccx': 3})


In [10]:
gate_counts = qc.count_ops()

non_measurement_gates = sum(
    count
    for gate, count in gate_counts.items()
    if gate != "measure"
)

print("Non-measurement gate count:", non_measurement_gates)

Non-measurement gate count: 12


In [11]:
print("===== BASELINE ADDER-CUM-SUBTRACTOR =====")
print("Qubits:", qc.num_qubits)
print("Depth:", qc.depth())
print("Gate count:", qc.count_ops())
print("Non-measurement gates:", non_measurement_gates)

===== BASELINE ADDER-CUM-SUBTRACTOR =====
Qubits: 5
Depth: 9
Gate count: OrderedDict({'cx': 6, 'measure': 5, 'x': 3, 'ccx': 3})
Non-measurement gates: 12
